# E2: What to do at the edge of the mask

**Question.** BiRefNet (chosen in E1) gives a soft alpha. Before compositing on the new colour, do we threshold it (hard edge), use it as it is (soft edge), or also replace the photo's colours at the edge by an estimate of the subject's own colour?

**Why it matters.** At an edge pixel the photo is already a mix, $I = \alpha F + (1-\alpha) B$ ($F$: the subject's colour, $B$: the old background). The current pipeline composites the photo itself:

$$\alpha I + (1-\alpha) C = \underbrace{\alpha F + (1-\alpha) C}_{\text{right answer}} + \underbrace{\alpha(1-\alpha)(B - F)}_{\text{old background leaking}}$$

The leak is largest at $\alpha = 0.5$ (hair, soft edges), and a perfect alpha does not remove it.

**Edge modes** (`bgchange/matting.py`, the code the CLI will use):

| Mode | Alpha | Colour composited |
|---|---|---|
| **H** `hard` | BiRefNet alpha thresholded at 0.5 | the photo |
| **S** `soft` | BiRefNet alpha | the photo (the current pipeline) |
| **S+F** `fg` | BiRefNet alpha | the foreground $\hat F$ estimated by pymatting `estimate_foreground_ml` (Germer et al., ICPR 2020), default settings |

ViTMatte refinement was considered and left out (time; BiRefNet's MAE on portraits is already 0.001 to 0.007 in E1).

**Data** (`data/e2_subset.md`, built by `data/make_e2_subset.py`):

| Group | n | Ground truth | Used for |
|---|---:|---|---|
| A_people (PPM-100) | 15 | soft alpha matte | the alpha: H vs S |
| A_synthetic | 30 | exact answer: the same 15 portraits as $F$ with their true alpha, on `#00FF00` and `#F0F0F0` | the colour: H vs S vs S+F |
| D_real | 6 | none | by eye: hair, sheer fabric, D06 (logo t-shirt) as a control |
| B_glass | 1 | binary mask, lenses marked opaque | by eye: an expected failure |

**Metrics.** Taken in the **edge region** of the ground truth: a band of 2 % of the image diagonal on both sides of the outline, plus every soft pixel. It is split into **soft** pixels ($0.02 < \alpha < 0.98$: in portraits, hair) and the **sharp** rest (the outline of the body and clothes).
- Alpha (A_people): MAE and gradient error (Rhemann et al., CVPR 2009) against the ground-truth matte.
- Colour (A_synthetic): mean absolute RGB error (grey levels, 0 to 255) against the exact answer on two new colours (`#1E90FF`, the brief's example, and white). Each mode is run with the **true alpha** (only the compositing error is left) and with **BiRefNet's alpha** (what the pipeline really does).
- Speed: seconds of foreground estimation per megapixel, on CPU (the CLI runs it on CPU).

**Hypotheses (written before the run)**
1. Most of BiRefNet's remaining alpha error on portraits is in the soft pixels (hair): their share of the edge error is larger than their share of the edge area.
2. H is worse than S on hair: higher MAE in the soft pixels and higher gradient error.
3. With the true alpha, S's colour error is the predicted leak $\alpha(1-\alpha)|B-F|$ (to 1.5 grey levels: rounding), larger on green than on near-white; S+F at least halves S's error in the soft pixels.
4. With BiRefNet's alpha, S+F still beats S on most images, by less (alpha errors add up).
5. Foreground estimation is cheap enough to be on by default: under 10 % of BiRefNet's CPU time per image (39 s for a 2.6 MP image in E1.0), so under 4 s for 2.6 MP.
6. Control (D06): inside the garment (BiRefNet alpha at least 0.99, 2 px or more from where it drops), S+F differs from S by at most 3 grey levels: it does not change the garment or its logo.
7. Expected failure (glass): inside the lenses the alpha is about 1, so $\hat F$ is the photo and the old background stays visible in every mode.

**How the notebook works.** Section 3 runs BiRefNet once per image and saves the alpha as a 16-bit PNG; sections 4 to 8 only read those alphas. Alphas and per-image metric rows already saved are skipped, so after a disconnect, run all again. After changing the metric code, delete `results/per_image/`.

**Run on Colab**
1. Commit and push: Colab clones the repo from GitHub (`bgchange/` and `data/e2_subset.csv` come from there).
2. Upload the content of `data/e2/` (206 MB: `A_people/`, `A_synthetic/`, `B_glass/`, `D_real/`) to Google Drive as `MyDrive/modelia-e2/`.
3. Connect to a GPU server (a T4 is enough) and run all. Optional first: `SMOKE = True` in the setup cell runs every cell on 6 images in a few minutes (results in `results/smoke/`).
4. Results go to `MyDrive/modelia-e2/results/`. Then share that Drive folder by link, paste the link into `experiments/fetch_e2_results.py` and run it locally: it copies the tables (and the figures, not committed) into `results/e2/`.

**Figures.** DIS-VD, PPM-100 (mattes CC BY-NC-SA 4.0) and Wikimedia Commons images (authors and licences in `data/e1_fashion_sources.csv`) are shown as in E1. D09 and D10 have no known licence: their figures are saved on Drive only, never shown here.

## 1. Setup

In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Colab runtimes ship the google.colab package (also with the VS Code extension, where it may not be imported yet).
IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/MedSahbiBenKalia/modelia-bg-change"
# Smoke test: every cell on 6 images (a few minutes), results in results/smoke/. Set True by hand, or E2_SMOKE=1.
SMOKE = os.environ.get("E2_SMOKE") == "1"


def run(cmd):
    # Like ! in a notebook, but a failure stops the notebook (a failed ! only prints and the next cells run anyway).
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{' '.join(map(str, cmd))} failed:\n{(r.stderr or r.stdout)[-3000:]}")


if IN_COLAB:
    ROOT = Path("/content/modelia-bg-change")
    run(["git", "-C", ROOT, "pull", "-q"] if ROOT.exists() else ["git", "clone", "-q", REPO_URL, ROOT])
    from importlib import invalidate_caches
    from importlib.metadata import version
    # Keep Colab's torch (a CUDA 13 build): pin the installed torch and torchvision, so that pip stops with an error
    # instead of silently replacing them with a build for another CUDA version.
    pins = Path("/content/e2_constraints.txt")
    pins.write_text(f"torch=={version('torch')}\ntorchvision=={version('torchvision')}\n")
    # pip may upgrade a package this Python has already imported: the old copy stays in memory and later imports
    # mix two versions (seen in E1: "cannot import name '_slice'"). Detect it.
    WATCH = {"numpy": "numpy", "scipy": "scipy", "pillow": "PIL", "pandas": "pandas", "numba": "numba"}  # pip name: import name
    before = {pkg: version(pkg) for pkg, mod in WATCH.items() if mod in sys.modules}
    run([sys.executable, "-m", "pip", "install", "-q", "-r", ROOT / "requirements.txt", "-c", pins])
    invalidate_caches()
    changed = [f"{pkg} {v} -> {version(pkg)}" for pkg, v in before.items() if version(pkg) != v]
    if changed:
        raise RuntimeError(f"pip upgraded {', '.join(changed)} after it was imported: "
                           "Runtime > Restart session, then run all again (the second run installs nothing).")
else:
    # Repo root = first parent folder that holds requirements.txt (works from experiments/ or the root).
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))
print("repo:", ROOT, "| colab:", IN_COLAB, "| smoke test:", SMOKE)

repo: /content/modelia-bg-change | colab: True | smoke test: False


In [2]:
import gc
import json
import platform
import shutil
import textwrap
import time
from importlib.metadata import version

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import spearmanr, wilcoxon

from bgchange.composite import over_colour
from bgchange.io import load_alpha, load_rgb, parse_hex
from bgchange.segment import Segmenter

try:
    from bgchange.matting import EDGE_MODES, hard_alpha, prepare
    from bgchange.metrics import edge_band, gradient_error, masked_mae, soft_pixels
except ImportError as e:
    raise ImportError(f"{e}. The bgchange/ in {ROOT} is older than this notebook: commit and push, then "
                      "Runtime > Restart session and run all (the setup cell pulls).") from None

MODE_LABEL = {"hard": "H", "soft": "S", "fg": "S+F"}


def cpu_name():
    # Readable CPU model for the speed table (platform.processor() is not, on Windows).
    try:
        if sys.platform == "win32":
            import winreg
            key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()
        return next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
    except Exception:
        return platform.processor() or "unknown CPU"


def cuda_works():
    """True if a small fp16 computation runs on the GPU.

    torch can see a GPU and still fail on it, at the first computation only. Colab's torch is a CUDA 13 build
    (cu130), and CUDA 13 dropped GPUs older than Turing (compute capability below 7.5, e.g. P100, V100):
    there it fails with "no kernel image is available for execution on the device".
    """
    if not torch.cuda.is_available():
        return False
    try:
        return (torch.ones(16, device="cuda", dtype=torch.float16) * 2).sum().item() == 32
    except Exception as e:
        try:
            major, minor = torch.cuda.get_device_capability()
            info = f"{torch.cuda.get_device_name()}, sm_{major}{minor}; this torch has kernels for {torch.cuda.get_arch_list()}"
        except Exception:
            info = "no device details"
        print(f"CUDA does not work here ({info}): {e!r}"[:800])
        print("Fix: connect to a T4, L4 or A100 server, or install a torch built for CUDA 12.6 "
              "(pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126) and restart.")
        return False


def md_table(df: pd.DataFrame, floatfmt="{:.3f}") -> str:
    # Markdown table without the tabulate dependency.
    cols = [df.index.name or ""] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        cells = [floatfmt.format(v) if isinstance(v, float) else str(v) for v in row]
        lines.append("| " + " | ".join([str(idx), *cells]) + " |")
    return "\n".join(lines)


DEVICE = "cuda" if cuda_works() else "cpu"
CPU = f"{cpu_name()} ({os.cpu_count()} threads)"
HARDWARE = torch.cuda.get_device_name() if DEVICE == "cuda" else CPU
print(f"BiRefNet on {DEVICE}: {HARDWARE} | foreground estimation on CPU: {CPU}")
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| transformers", version("transformers"),
      "| pymatting", version("pymatting"), "| numba", version("numba"), "| numpy", np.__version__)
if IN_COLAB and DEVICE == "cpu":
    print("WARNING: BiRefNet will run on CPU, minutes per image. Pick a GPU server (a T4 is enough).")

ImportError: cannot import name 'EDGE_MODES' from 'bgchange.matting' (/content/modelia-bg-change/bgchange/matting.py). The bgchange/ in /content/modelia-bg-change is older than this notebook: commit and push, then Runtime > Restart session and run all (the setup cell pulls).

### Data and results folders

On Colab the images come from Google Drive (`MyDrive/modelia-e2/`, the content of `data/e2/`). They are copied once to the Colab disk (`/content/e2_data/`): faster than reading through the Drive mount, and no Drive error in the middle of the run. Results are written to `modelia-e2/results/` on Drive, so they survive a disconnect. If mounting Drive fails (it needs a Google login pop-up), paste the folder's share link into `DRIVE_FOLDER_URL`: the images are then downloaded with `gdown` and the results stay on the Colab disk only.

Locally: `data/e2/` and `results/e2/`.

Every results folder holds at most 30 files: `gdown` (used by `fetch_e2_results.py`) lists at most 50 files per folder.

In [ ]:
DRIVE_FOLDER = "modelia-e2"  # MyDrive/modelia-e2/ = the content of data/e2/ (A_people/, A_synthetic/, ...)
DRIVE_FOLDER_URL = ""        # optional: its share link, only used if mounting Drive fails


def find_e2_root(base: Path) -> Path:
    # The folder that holds A_people/, A_synthetic/, ... (the upload may have added one level, e.g. modelia-e2/e2/).
    for cand in [base, *base.glob("*"), *base.glob("*/*")]:
        if (cand / "A_synthetic").is_dir():
            return cand
    raise FileNotFoundError(f"no A_synthetic/ folder under {base}: upload the content of data/e2/ there")


if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        mydrive = Path("/content/drive/MyDrive")
        base = next((p for p in [mydrive / DRIVE_FOLDER, *mydrive.glob(f"*/{DRIVE_FOLDER}")] if p.is_dir()), None)
        if base is None:
            raise FileNotFoundError(f"MyDrive/{DRIVE_FOLDER}/ not found: upload the content of data/e2/ to Google Drive under that name")
        RESULTS = base / "results"
    except Exception as e:
        if not DRIVE_FOLDER_URL:
            print("Drive could not be used. Run in Colab in the browser, or share the folder by link and set DRIVE_FOLDER_URL.")
            raise
        print("Drive not used:", repr(e), "-> downloading the shared folder with gdown")
        import gdown
        base = Path("/content") / DRIVE_FOLDER
        if not (base.exists() and any(base.iterdir())):
            gdown.download_folder(DRIVE_FOLDER_URL, output=str(base), quiet=True)
        RESULTS = Path("/content/e2_results")
        print("WARNING: results are only on the Colab disk, download them before the session ends:", RESULTS)
    E2_ROOT = Path("/content/e2_data")
    if not (E2_ROOT / "A_synthetic").is_dir():
        t0 = time.perf_counter()
        partial = Path("/content/e2_data_partial")
        shutil.rmtree(partial, ignore_errors=True)
        shutil.copytree(find_e2_root(base), partial, ignore=shutil.ignore_patterns("results"))
        partial.rename(E2_ROOT)  # only a complete copy gets the final name
        print(f"copied the images to {E2_ROOT} in {time.perf_counter() - t0:.0f} s")
else:
    E2_ROOT = ROOT / "data" / "e2"
    RESULTS = ROOT / "results" / "e2"
if SMOKE:
    RESULTS = RESULTS / "smoke"
ALPHAS = RESULTS / "alphas"          # alphas/<group>/<name>.png, 16-bit
PER_IMAGE = RESULTS / "per_image"    # per_image/<alpha|colour>/<name>.csv
FIGURES = RESULTS / "figures"
COMPOSITES = RESULTS / "composites"  # composites/<mode>/<name>_<colour>.jpg, real photos only
RESULTS.mkdir(parents=True, exist_ok=True)
print("images :", E2_ROOT)
print("results:", RESULTS)

In [ ]:
subset = pd.read_csv(ROOT / "data" / "e2_subset.csv", keep_default_na=False)


def local(rel: str) -> Path:
    # CSV paths are 'e2/<group>/...' relative to data/.
    return E2_ROOT / Path(rel).relative_to("e2")


missing = sorted({p for col in ("image", "gt", "fg") for p in subset[col] if p and not local(p).exists()})
assert not missing, (f"{len(missing)} files missing under {E2_ROOT}, e.g. {missing[:3]}. Upload not finished? "
                     "On Colab, then delete /content/e2_data and run the cell above again.")

if SMOKE:
    # The smallest zoom portrait and its two synthetic versions, the smallest real photo, D06 (the control) and the glass image.
    size = subset["image"].map(lambda p: local(p).stat().st_size)
    zoom = subset[(subset["group"] == "A_people") & subset["note"].str.startswith("zoom")]
    person = zoom.loc[size[zoom.index].idxmin()]
    keep = (subset["name"] == person["name"]) | (subset["fg"] == person["image"]) | (subset["group"] == "B_glass")
    keep |= (subset.index == size[subset["group"] == "D_real"].idxmin()) | (subset["name"] == "D06_logo_tshirt")
    subset = subset[keep].reset_index(drop=True)

people = subset[subset["group"] == "A_people"]
synthetic = subset[subset["group"] == "A_synthetic"]
real = subset[subset["group"].isin(["D_real", "B_glass"])]

ENV = {"device": DEVICE, "hardware": HARDWARE, "cpu": CPU, "python": platform.python_version(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "numpy": np.__version__, "opencv": cv2.__version__,
       **{p: version(p) for p in ("transformers", "pymatting", "numba")}}
(RESULTS / "e2_environment.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")
print(subset.groupby("group").size().to_string())

## 2. Where are the soft pixels? (ground truth only, no model)

For each portrait: the share of soft pixels ($0.02 < \alpha < 0.98$) in the ground-truth matte, and BiRefNet's MAE on that image in E1 (`results/e1/e1_metrics.csv`). If BiRefNet's error comes from the soft pixels, images with more of them should have a higher MAE. The maps show where the soft pixels are.

In [ ]:
e1 = pd.read_csv(ROOT / "results" / "e1" / "e1_metrics.csv")
E1_MAE = e1[(e1["model"] == "BiRefNet") & (e1["group"] == "A_people")].set_index("name")["mae"]


def thumb(a, size=512):
    """Thumbnail of an RGB image or a [0, 1] map, as an array, at most `size` px on its long side."""
    im = Image.fromarray(a if np.asarray(a).dtype == np.uint8 else (np.clip(a, 0, 1) * 255).astype(np.uint8))
    im.thumbnail((size, size))
    return np.asarray(im)


rows, soft_maps = [], {}
for r in people.to_dict("records"):
    gt = load_alpha(local(r["gt"]))
    soft = soft_pixels(gt)
    rows.append({"name": r["name"], "megapixels": gt.size / 1e6, "soft_pct": 100 * soft.mean(),
                 "soft_per_subject_pct": 100 * soft.sum() / max((gt >= 0.5).sum(), 1),
                 "e1_birefnet_mae": E1_MAE.get(r["name"], np.nan)})
    soft_maps[r["name"]] = thumb(soft.astype(np.float32), 300)
soft_share = pd.DataFrame(rows).sort_values("soft_pct", ascending=False).set_index("name")
soft_share.to_csv(RESULTS / "e2_soft_share.csv")
display(soft_share.round(4))
if len(soft_share) >= 5:
    rho, p = spearmanr(soft_share["soft_pct"], soft_share["e1_birefnet_mae"])
    print(f"Spearman(soft % of the image, BiRefNet MAE in E1) = {rho:.2f}, p = {p:.2g}, n = {len(soft_share)}")

cols = 5
n_rows = -(-len(soft_maps) // cols)  # ceiling division
if soft_maps:
    fig, axes = plt.subplots(n_rows, cols, figsize=(3 * cols, 2.6 * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, name in zip(axes.flat, soft_share.index):
        ax.imshow(soft_maps[name], cmap="gray")
        ax.set_title(f"{name.split('_')[0]}: {soft_share.loc[name, 'soft_pct']:.1f} % soft", fontsize=8)
    fig.suptitle("Soft pixels of the ground-truth mattes (white), most soft first", fontsize=10)
    plt.tight_layout()
    FIGURES.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES / "e2_soft_maps.png", dpi=110)
    plt.show()

## 3. BiRefNet alphas (GPU)

`bgchange.segment.Segmenter`: the pipeline's own code, BiRefNet at the revision pinned after E1, fp16 on the GPU. One warm-up image (not timed), then every image is timed from PIL image in to full-size alpha out. The alpha is saved as a 16-bit PNG in `alphas/<group>/`; an image already done is skipped.

In [ ]:
def alpha_path(r):
    return ALPHAS / r["group"] / f"{r['name']}.png"


def save_alpha16(alpha, path):
    # 16-bit PNG: 8 bits round each pixel by up to 0.002, the size of BiRefNet's MAE on portraits in E1 (0.001-0.007).
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.stem + ".partial.png")
    if not cv2.imwrite(str(partial), (np.clip(alpha, 0, 1) * 65535).round().astype(np.uint16)):
        raise OSError(f"could not write {partial}")
    partial.replace(path)  # a disconnect mid-write leaves a .partial file, never a broken alpha that would be skipped


def load_alpha16(path):
    a = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if a is None:
        raise FileNotFoundError(f"no alpha at {path}: run section 3 first")
    return a.astype(np.float32) / 65535


timing_csv = RESULTS / "e2_birefnet_timing.csv"
todo = [r for r in subset.to_dict("records") if not alpha_path(r).exists()]
print(f"{len(subset) - len(todo)} alphas already done, {len(todo)} to compute on {DEVICE}")
if todo:
    t0 = time.perf_counter()
    try:
        segmenter = Segmenter(DEVICE)
    except Exception as e:
        raise RuntimeError(f"BiRefNet did not load with transformers {version('transformers')}: {e!r}. If the error comes "
                           "from the model's own code, pip install 'transformers<5', restart and run all.") from e
    load_s = time.perf_counter() - t0
    segmenter(load_rgb(local(todo[0]["image"])))  # warm-up, not timed (first-call overheads, CUDA kernel selection)
    for i, r in enumerate(todo, 1):
        img = load_rgb(local(r["image"]))
        t = time.perf_counter()
        alpha = segmenter(img)  # returns a CPU array, so the GPU work is finished when the clock stops
        row = {"name": r["name"], "group": r["group"], "megapixels": img.width * img.height / 1e6,
               "seconds": time.perf_counter() - t, "device": DEVICE, "hardware": HARDWARE, "load_seconds": load_s}
        save_alpha16(alpha, alpha_path(r))
        pd.DataFrame([row]).to_csv(timing_csv, mode="a", header=not timing_csv.exists(), index=False)
        print(f"{i}/{len(todo)} {r['name']}: {row['seconds']:.2f} s")
    del segmenter
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 4. Alpha at the edge: H vs S (A_people, 15 portraits)

Per portrait, BiRefNet's soft alpha (S) and the same alpha thresholded at 0.5 (H) against the ground-truth matte: MAE in the edge region and in its soft (hair) and sharp parts, gradient error in the edge region, and the share of the edge's alpha error that lies in soft pixels next to their share of the edge area (hypothesis 1).

Check first: the model and the images are the ones of E1, so the full-image MAE must match E1's.

In [ ]:
def regions(gt):
    """edge = band on both sides of the outline + every soft pixel; soft = 0.02 < gt < 0.98 (hair); sharp = edge - soft."""
    soft = soft_pixels(gt)
    edge = edge_band(gt) | soft
    return {"edge": edge, "soft": soft, "sharp": edge & ~soft}


def write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_name(path.name + ".partial")
    df.to_csv(partial, index=False)
    partial.replace(path)


def cached_rows(kind, name, compute):
    """Metric rows of one image, saved once: a re-run after a disconnect skips the images already done."""
    path = PER_IMAGE / kind / f"{name}.csv"
    if path.exists():
        return pd.read_csv(path).to_dict("records")
    rows = compute()
    write_csv(pd.DataFrame(rows), path)
    return rows


def alpha_rows(r):
    gt = load_alpha(local(r["gt"]))
    pred = load_alpha16(alpha_path(r))
    assert pred.shape == gt.shape, f"{r['name']}: alpha {pred.shape} vs ground truth {gt.shape}"
    reg = regions(gt)
    rows = []
    for mode, a in (("soft", pred), ("hard", hard_alpha(pred))):
        err = np.abs(a - gt)
        row = {"name": r["name"], "alpha": mode, "mae_image": float(err.mean()),
               "grad_edge": gradient_error(a, gt, reg["edge"]),
               "soft_error_share": float(err[reg["soft"]].sum() / max(err[reg["edge"]].sum(), 1e-9)),
               "soft_area_share": float(reg["soft"].sum() / reg["edge"].sum())}
        row.update({f"mae_{k}": masked_mae(a, gt, m) for k, m in reg.items()})
        rows.append(row)
    return rows


t0 = time.perf_counter()
alpha_metrics = pd.DataFrame([row for r in people.to_dict("records")
                              for row in cached_rows("alpha", r["name"], lambda r=r: alpha_rows(r))])
alpha_metrics.to_csv(RESULTS / "e2_alpha_metrics.csv", index=False)
print(f"{len(people)} portraits in {time.perf_counter() - t0:.0f} s")

gap_e1 = (alpha_metrics[alpha_metrics["alpha"] == "soft"].set_index("name")["mae_image"] - E1_MAE).abs().max()
print(f"check: full-image MAE vs E1, largest difference {gap_e1:.4f}"
      + ("" if gap_e1 < 0.002 else "  <- more than rounding: another model revision, or a rotated image?"))

ALPHA_COLS = {"mae_edge": "MAE edge", "mae_soft": "MAE soft (hair)", "mae_sharp": "MAE sharp (body, clothes)",
              "grad_edge": "gradient error edge (x1000)", "soft_area_share": "soft share of the edge area",
              "soft_error_share": "soft share of the edge error"}
alpha_summary = alpha_metrics.groupby("alpha")[list(ALPHA_COLS)].mean().reindex(["soft", "hard"])
alpha_summary["grad_edge"] *= 1000
alpha_summary = alpha_summary.rename(columns=ALPHA_COLS, index={"soft": "S (BiRefNet alpha)", "hard": "H (thresholded at 0.5)"})
alpha_summary.index.name = f"mean over {len(people)} portraits"
(RESULTS / "e2_alpha_summary.md").write_text(md_table(alpha_summary, "{:.4f}"), encoding="utf-8")
display(alpha_summary.round(4))

## 5. Colour at the edge: H vs S vs S+F against the exact answer (A_synthetic, 30 images)

Each synthetic input is $X = \alpha F + (1-\alpha) B$, so the right result on a new colour $C$ is known: $\alpha F + (1-\alpha) C$. For each image, alpha source (true / BiRefNet), mode and new colour, the error is the mean absolute RGB difference to that answer, in grey levels.

Check first (hypothesis 3, and a test of this code): with the true alpha, S's error must be the predicted leak $\alpha(1-\alpha)|B-F|$ at every pixel, up to rounding (1.5 grey levels).

The first foreground estimation compiles pymatting's numba code (or loads it from numba's disk cache): it is timed apart.

In [ ]:
NEW_COLOURS = ["#1E90FF", "#FFFFFF"]  # the brief's example colour, and white (the usual e-commerce background)

t = time.perf_counter()
prepare(Image.new("RGB", (64, 64)), np.full((64, 64), 0.5, np.float32), "fg")
FG_FIRST_CALL_S = time.perf_counter() - t
print(f"first foreground estimation (numba compile or cache load): {FG_FIRST_CALL_S:.1f} s")


def formula_gap(out, truth, f, gt, background):
    """Largest gap, over all pixels and channels, between S's measured error (true alpha) and the predicted
    leak a(1 - a)|B - F|. The input, the output and the answer are each rounded to 8 bits: up to 1.5 levels."""
    a = gt[..., None]
    leak = a * (1 - a) * np.abs(np.array(parse_hex(background), np.float32) - np.asarray(f, np.float32))
    measured = np.abs(np.asarray(out, np.float32) - np.asarray(truth, np.float32))
    return float(np.abs(measured - leak).max())


def colour_rows(r):
    x = load_rgb(local(r["image"]))  # the synthetic input: alpha * F + (1 - alpha) * B
    f = load_rgb(local(r["fg"]))     # F, the true foreground
    gt = load_alpha(local(r["gt"]))  # the true alpha
    pred = load_alpha16(alpha_path(r))
    assert x.size == f.size and pred.shape == gt.shape == (x.height, x.width), f"{r['name']}: sizes differ"
    reg = regions(gt)
    truth = {c: over_colour(f, gt, parse_hex(c)) for c in NEW_COLOURS}  # the exact answer on each new colour
    base = {"name": r["name"], "portrait": Path(r["fg"]).stem, "background": r["background"],
            "megapixels": x.width * x.height / 1e6}
    rows = []
    for source, a in (("true", gt), ("birefnet", pred)):
        for mode in EDGE_MODES:
            t = time.perf_counter()
            img, a_used = prepare(x, a, mode)
            seconds = time.perf_counter() - t
            for c in NEW_COLOURS:
                out = over_colour(img, a_used, parse_hex(c))
                gap = formula_gap(out, truth[c], f, gt, r["background"]) if (source, mode) == ("true", "soft") else np.nan
                rows += [{**base, "alpha": source, "mode": mode, "colour": c, "region": k,
                          "error": masked_mae(out, truth[c], m), "prepare_seconds": seconds, "formula_gap": gap}
                         for k, m in reg.items()]
    return rows


t0 = time.perf_counter()
colour = []
for i, r in enumerate(synthetic.to_dict("records"), 1):
    colour += cached_rows("colour", r["name"], lambda r=r: colour_rows(r))
    print(f"{i}/{len(synthetic)} {r['name']} ({time.perf_counter() - t0:.0f} s)")
colour = pd.DataFrame(colour)
colour.to_csv(RESULTS / "e2_colour_metrics.csv", index=False)

gap = colour["formula_gap"].max()
assert gap <= 1.5 + 1e-3, f"S with the true alpha is {gap:.2f} levels off the predicted leak: a bug in this test"
print(f"check passed: S with the true alpha = the predicted leak a(1-a)|B-F|, to {gap:.2f} grey levels (rounding allows 1.5)")

In [ ]:
def colour_table(c):
    """Mean error over the portraits for one new colour: rows = alpha source and background, columns = region and mode."""
    d = colour[(colour["colour"] == c) & colour["region"].isin(["soft", "edge"])]
    t = d.pivot_table(index=["alpha", "background"], columns=["region", "mode"], values="error", aggfunc="mean")
    t = t.reindex(columns=pd.MultiIndex.from_product([["soft", "edge"], EDGE_MODES]))
    t = t.loc[[s for s in ("true", "birefnet") if s in t.index.get_level_values(0)]]
    t.columns = [f"{region} {MODE_LABEL[m]}" for region, m in t.columns]
    t.index = [f"{'true' if s == 'true' else 'BiRefNet'} alpha, old background {b}" for s, b in t.index]
    t.index.name = f"new colour {c}"
    return t


def paired(d):
    """S+F vs S on the same portraits: mean errors, wins, and a Wilcoxon signed-rank test (paired, no normality)."""
    w = d.pivot(index="portrait", columns="mode", values="error")
    diff = w["fg"] - w["soft"]
    p = wilcoxon(diff).pvalue if len(diff) >= 5 and (diff != 0).any() else np.nan
    return {"n": len(diff), "S": w["soft"].mean(), "S+F": w["fg"].mean(), "S+F / S": w["fg"].mean() / w["soft"].mean(),
            "S+F better on": int((diff < 0).sum()), "wilcoxon p": p}


lines = ["Mean absolute RGB error against the exact answer, in grey levels (0-255), mean over the portraits.", ""]
for c in NEW_COLOURS:
    t = colour_table(c)
    display(t.round(2))
    lines += [md_table(t, "{:.2f}"), ""]
(RESULTS / "e2_colour_summary.md").write_text("\n".join(lines), encoding="utf-8")

soft_rows = colour[colour["region"] == "soft"]
paired_table = pd.DataFrame([{"colour": c, "alpha": s, "old background": b, **paired(d)}
                             for (c, s, b), d in soft_rows.groupby(["colour", "alpha", "background"])])
paired_table = paired_table.set_index(["colour", "alpha", "old background"])
shown = paired_table.reset_index().astype({"wilcoxon p": object})
shown["wilcoxon p"] = paired_table["wilcoxon p"].map(lambda p: f"{p:.2g}").to_numpy()
shown = shown.set_index("colour")
(RESULTS / "e2_paired.md").write_text("S+F vs S in the soft pixels, per portrait (lower error is better).\n\n"
                                      + md_table(shown, "{:.3f}"), encoding="utf-8")
display(paired_table.round(3))

## 6. Figures: portraits

Zoomed crops (lossless, from the full-size arrays). The window is chosen by rule, not by eye: where BiRefNet's alpha is furthest from the ground truth (real portraits), or where S is furthest from the right answer (synthetic). Error maps: black = no error, bright = 64 grey levels or more (alpha maps: 0.5 or more).

In [ ]:
HEAT_MAX = 64  # grey levels: top of the colour scale of the colour-error maps
FIG_COLOUR = "#1E90FF"


def window(score, frac=0.12):
    """Top-left corner and side of the square window with the highest mean score (side: 12 % of the short side, 96-512 px)."""
    k = min(int(np.clip(frac * min(score.shape), 96, 512)), *score.shape)
    smooth = cv2.blur(score.astype(np.float32), (k, k))  # mean of the k x k window centred on each pixel
    cy, cx = np.unravel_index(smooth.argmax(), smooth.shape)
    return int(np.clip(cy - k // 2, 0, score.shape[0] - k)), int(np.clip(cx - k // 2, 0, score.shape[1] - k)), k


def crop(p, win):
    y0, x0, k = win
    return np.asarray(p)[y0:y0 + k, x0:x0 + k]


def diff_map(a, b):
    """Per-pixel |a - b| of two RGB images, mean over the channels, in grey levels."""
    return np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean(-1)


def show_grid(rows, titles, path, show=True, suptitle=None):
    """rows: (label, panels, notes). A panel is an RGB array, a 2-D alpha in [0, 1] (grey), or ("heat", map, vmax)."""
    fig, axes = plt.subplots(len(rows), len(titles), figsize=(2.3 * len(titles), 2.6 * len(rows)), squeeze=False)
    for i, (label, panels, notes) in enumerate(rows):
        for j, (ax, p) in enumerate(zip(axes[i], panels)):
            if isinstance(p, tuple):
                ax.imshow(p[1], cmap="magma", vmin=0, vmax=p[2])
            elif np.ndim(p) == 2:
                ax.imshow(p, cmap="gray", vmin=0, vmax=255 if np.asarray(p).dtype == np.uint8 else 1)
            else:
                ax.imshow(p)
            ax.set_xticks([]), ax.set_yticks([])
            if i == 0:
                ax.set_title(titles[j], fontsize=8)
            if notes and notes[j]:
                ax.set_xlabel(notes[j], fontsize=7)
        axes[i][0].set_ylabel(label, fontsize=7)
    if suptitle:
        fig.suptitle(suptitle, fontsize=9)
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=110)
    plt.show() if show else plt.close(fig)


def short(r):
    # Row label: the image number, then why it was picked, wrapped to fit next to the row.
    return r["name"].split("_")[0] + "\n" + textwrap.fill(r["note"].replace("zoom", "").strip(":, "), 30)


zoom_people = people[people["note"].str.startswith("zoom")]

# Real portraits: the alpha
grid_rows = []
for r in zoom_people.to_dict("records"):
    img, gt, pred = load_rgb(local(r["image"])), load_alpha(local(r["gt"])), load_alpha16(alpha_path(r))
    hard = hard_alpha(pred)
    win = window(np.abs(pred - gt))
    m = alpha_metrics[alpha_metrics["name"] == r["name"]].set_index("alpha")
    grid_rows.append((short(r), [crop(img, win), crop(gt, win), crop(pred, win), crop(hard, win),
                                 ("heat", crop(np.abs(pred - gt), win), 0.5), ("heat", crop(np.abs(hard - gt), win), 0.5)],
                      ["", "", f"MAE soft {m.loc['soft', 'mae_soft']:.3f}", f"MAE soft {m.loc['hard', 'mae_soft']:.3f}", "", ""]))
if grid_rows:
    show_grid(grid_rows, ["photo", "ground truth", "S: BiRefNet alpha", "H: thresholded", "|S - truth|", "|H - truth|"],
              FIGURES / "e2_A_alpha_zoom.png", suptitle="Real portraits (PPM-100): the alpha, where BiRefNet is furthest from the ground truth")

# Synthetic portraits: the colour, with BiRefNet's alpha (what the pipeline does)
rgb = parse_hex(FIG_COLOUR)
for bg in sorted(synthetic["background"].unique()):
    grid_rows = []
    for r in synthetic[(synthetic["background"] == bg) & synthetic["fg"].isin(zoom_people["image"])].to_dict("records"):
        x, f = load_rgb(local(r["image"])), load_rgb(local(r["fg"]))
        gt, pred = load_alpha(local(r["gt"])), load_alpha16(alpha_path(r))
        truth = over_colour(f, gt, rgb)
        outs = {m: over_colour(*prepare(x, pred, m), rgb) for m in EDGE_MODES}
        err = {m: diff_map(outs[m], truth) for m in ("soft", "fg")}
        win = window(err["soft"])
        e = colour[(colour["name"] == r["name"]) & (colour["alpha"] == "birefnet") & (colour["colour"] == FIG_COLOUR)
                   & (colour["region"] == "soft")].set_index("mode")["error"]
        grid_rows.append((short(zoom_people[zoom_people["image"] == r["fg"]].iloc[0]),
                          [crop(x, win), crop(truth, win), *(crop(outs[m], win) for m in EDGE_MODES),
                           ("heat", crop(err["soft"], win), HEAT_MAX), ("heat", crop(err["fg"], win), HEAT_MAX)],
                          ["", "", *(f"soft error {e[m]:.1f}" for m in EDGE_MODES), "", ""]))
    if grid_rows:
        show_grid(grid_rows, ["input", f"right answer on {FIG_COLOUR}", "H", "S", "S+F", "|S - answer|", "|S+F - answer|"],
                  FIGURES / f"e2_A_synthetic_{bg[1:]}.png",
                  suptitle=f"Synthetic: portraits on {bg} moved to {FIG_COLOUR}, BiRefNet alpha (soft error = mean in the hair, grey levels)")

## 7. Real photos (D), the control (D06) and glass

No ground truth: judged by eye. Each mode on white (shows a dark leak, e.g. D10's brown background) and near-black (shows a light leak, e.g. D05's and D09's white backgrounds). The zoom window is where BiRefNet's alpha is softest (hair, sheer fabric). Full composites (at most 1600 px) are saved to `composites/<mode>/`.

D06 control (hypothesis 6): S and S+F on white, the largest difference inside the garment.

In [ ]:
FIGURE_COLOURS = ["#FFFFFF", "#202020"]


def save_jpeg(img, path, size=1600):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(thumb(np.asarray(img), size)).save(path, quality=92)


def max_thumb(m, size=512):
    """Thumbnail of a difference map that keeps small spots visible: max over each block, not the mean."""
    s = int(np.ceil(max(m.shape) / size))
    return cv2.dilate(m.astype(np.float32), np.ones((s, s), np.uint8))[::s, ::s] if s > 1 else m


for r in real.to_dict("records"):
    img, pred = load_rgb(local(r["image"])), load_alpha16(alpha_path(r))
    win = window(soft_pixels(pred).astype(np.float32))
    y0, x0, k = win
    overview = np.asarray(img).copy()
    cv2.rectangle(overview, (x0, y0), (x0 + k, y0 + k), (255, 0, 0), max(2, min(img.size) // 150))
    panels, titles = [thumb(overview), crop(img, win), crop(pred, win)], ["photo (box: zoom)", "zoom", "BiRefNet alpha"]
    prepared = {m: prepare(img, pred, m) for m in EDGE_MODES}
    for c in FIGURE_COLOURS:
        for m in EDGE_MODES:
            out = over_colour(*prepared[m], parse_hex(c))
            save_jpeg(out, COMPOSITES / m / f"{r['name']}_{c[1:]}.jpg")
            panels.append(crop(out, win))
            titles.append(f"{MODE_LABEL[m]} on {c}")
    show = "unknown" not in r["source"]
    if not show:
        print(f"{r['name']}: no known licence, figure saved to {FIGURES / 'real'} only")
    show_grid([(r["name"], panels, None)], titles, FIGURES / "real" / f"{r['name']}.png", show=show, suptitle=r["note"])

    if r["name"] == "D06_logo_tshirt":
        white = (255, 255, 255)
        s = np.asarray(over_colour(*prepared["soft"], white), np.int16)
        sf = np.asarray(over_colour(*prepared["fg"], white), np.int16)
        d = np.abs(sf - s).max(-1)
        # alpha >= 0.99 and 2 px or more from where it drops: away from the 1-px ring where pymatting smooths
        inside = cv2.erode((pred >= 0.99).astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
        control = {"pixels inside the garment": int(inside.sum()), "max |S+F - S| inside": int(d[inside].max()),
                   "99.9th percentile inside": float(np.percentile(d[inside], 99.9)),
                   "max |S+F - S| whole image": int(d.max()), "mean |S+F - S| whole image": float(d.mean())}
        (RESULTS / "e2_control_D06.json").write_text(json.dumps(control, indent=2), encoding="utf-8")
        print("D06 control (grey levels):", control)
        show_grid([("D06", [thumb(np.asarray(img)), thumb(pred), ("heat", max_thumb(d.astype(np.float32)), 10)], None)],
                  ["photo", "BiRefNet alpha", "|S+F - S| on white (0-10 levels)"], FIGURES / "e2_D06_control.png")

    if r["group"] == "B_glass":
        on = parse_hex(FIG_COLOUR)
        show_grid([("glass", [thumb(np.asarray(img)), thumb(pred), thumb(np.asarray(over_colour(*prepared["soft"], on))),
                              thumb(np.asarray(over_colour(*prepared["fg"], on)))], None)],
                  ["photo", "BiRefNet alpha", f"S on {FIG_COLOUR}", f"S+F on {FIG_COLOUR}"], FIGURES / "e2_glass.png",
                  suptitle="Glass: is the old background still visible inside the lenses?")

## 8. Speed

BiRefNet: median seconds per image on the device above. Foreground estimation (S+F): median seconds per image and per megapixel on CPU, from the A_synthetic runs (it grows with the pixel count), and the first call of the session apart (numba compile or cache load).

In [ ]:
bt = pd.read_csv(timing_csv) if timing_csv.exists() else pd.DataFrame(columns=["device", "hardware", "megapixels", "seconds", "load_seconds"])
fg_t = (colour[colour["mode"] == "fg"].groupby(["name", "alpha"])
        .agg(megapixels=("megapixels", "first"), seconds=("prepare_seconds", "first")).reset_index())
speed = [{"step": f"BiRefNet on {dev}", "hardware": hw, "images": len(d), "median megapixels": d["megapixels"].median(),
          "median s per image": d["seconds"].median(), "s per megapixel": np.nan, "load s": d["load_seconds"].iloc[0]}
         for (dev, hw), d in bt.groupby(["device", "hardware"])]
speed.append({"step": "foreground estimation (S+F)", "hardware": CPU, "images": len(fg_t),
              "median megapixels": fg_t["megapixels"].median(), "median s per image": fg_t["seconds"].median(),
              "s per megapixel": (fg_t["seconds"] / fg_t["megapixels"]).median(), "load s": FG_FIRST_CALL_S})
speed = pd.DataFrame(speed).set_index("step")
speed["s for 2.6 MP"] = speed["s per megapixel"] * 2.6
speed.to_csv(RESULTS / "e2_speed.csv")
(RESULTS / "e2_speed.md").write_text(md_table(speed, "{:.2f}") + "\n\n`load s`: loading BiRefNet, or the first "
                                     "foreground estimation of the session (numba compile or cache load).", encoding="utf-8")
display(speed.round(2))

## 9. The numbers for each hypothesis

Measured values only: the verdicts are written in section 10 and in `EXPERIMENTS.md` after looking at these and at the figures.

In [ ]:
S = alpha_metrics[alpha_metrics["alpha"] == "soft"].set_index("name")
H = alpha_metrics[alpha_metrics["alpha"] == "hard"].set_index("name")
true_soft = soft_rows[(soft_rows["alpha"] == "true") & (soft_rows["colour"] == NEW_COLOURS[0])]
by_bg = true_soft.pivot_table(index="background", columns="mode", values="error", aggfunc="mean")
fg_s = speed.loc["foreground estimation (S+F)", "s for 2.6 MP"]
out = [
    f"H1 soft pixels are {100 * S['soft_area_share'].mean():.0f} % of the edge area and hold "
    f"{100 * S['soft_error_share'].mean():.0f} % of BiRefNet's alpha error there (mean over {len(S)} portraits; "
    f"error share > area share on {(S['soft_error_share'] > S['soft_area_share']).sum()}/{len(S)})",
    f"H2 MAE in the soft pixels: S {S['mae_soft'].mean():.4f} vs H {H['mae_soft'].mean():.4f} (S lower on "
    f"{(S['mae_soft'] < H['mae_soft']).sum()}/{len(S)}); gradient error x1000: S {1000 * S['grad_edge'].mean():.3f} "
    f"vs H {1000 * H['grad_edge'].mean():.3f} (S lower on {(S['grad_edge'] < H['grad_edge']).sum()}/{len(S)})",
    f"H3 formula check: largest gap {colour['formula_gap'].max():.2f} levels; true alpha, soft pixels, {NEW_COLOURS[0]}: "
    + "; ".join(f"on {bg} S {row['soft']:.2f} vs S+F {row['fg']:.2f} (S+F/S {row['fg'] / row['soft']:.2f})"
                for bg, row in by_bg.iterrows()),
    "H4 BiRefNet alpha, soft pixels, S+F better than S on: "
    + "; ".join(f"{c} on {b}: {int(v['S+F better on'])}/{int(v['n'])} (S {v['S']:.2f}, S+F {v['S+F']:.2f}, p {v['wilcoxon p']:.2g})"
                for (c, s, b), v in paired_table.iterrows() if s == "birefnet"),
    f"H5 foreground estimation: {fg_s:.2f} s for 2.6 MP on {CPU} (threshold 3.9 s; first call {FG_FIRST_CALL_S:.1f} s)",
]
control_path = RESULTS / "e2_control_D06.json"
if control_path.exists():
    ctl = json.loads(control_path.read_text(encoding="utf-8"))
    out.append(f"H6 D06: max |S+F - S| inside the garment {ctl['max |S+F - S| inside']} levels "
               f"(99.9th percentile {ctl['99.9th percentile inside']:.1f}; threshold 3)")
out.append("H7 glass: by eye, figures/e2_glass.png")
(RESULTS / "e2_hypotheses_numbers.md").write_text("\n".join(f"- {line}" for line in out), encoding="utf-8")
print("\n".join(out))

## 10. Result and decision

Written after the run, from section 9 and the figures (full entry: `EXPERIMENTS.md`, E2).